# ACV Exploratory Data Analysis

This notebook investigates differences between the labelled faulty car and the remaining cars in each ACV training case.

Before beginning exploratory analysis, the reusable ACV ingestion functions are tested to verify that files, car identifiers, and telemetry parameters can be discovered dynamically.

In [18]:
from pathlib import Path
import sys
import pandas as pd

CODE_ROOT = Path("..").resolve()
REPO_ROOT = Path("../../../..").resolve()

if str(CODE_ROOT) not in sys.path:
    sys.path.append(str(CODE_ROOT))

from src.ingestion import (
    load_acv_file,
    get_car_ids,
    get_parameters,
    wide_to_long,
)

from src.preprocessing import preprocess_acv

## 1. Ingestion Sanity Check

Load a standard-schema training case and verify that the reusable ingestion functions correctly identify its cars and ACV parameters.

In [19]:
case_01_path = REPO_ROOT / "data" / "ACV" / "Train" / "acv_case_01.xlsx"

case_01 = load_acv_file(case_01_path)

print("Shape:", case_01.shape)
print("Cars:", get_car_ids(case_01))

print("\nParameters:")
for parameter in get_parameters(case_01):
    print(f"  {parameter}")

Shape: (6999, 67)
Cars: ['01', '02', '03', '04', '05', '06', '07', '08']

Parameters:
  ACV Control Temperature (Cooling)
  ACV Control Temperature (Heating)
  ACV Information Valid
  ACV Running Mode
  ACV Setting Mode
  Indoor Average Temperature
  Load Halved
  Outdoor Average Temperature


## 2. Wide-to-Long Transformation

The raw ACV files store telemetry for all eight cars in separate columns. For analysis and modelling, the dataset is transformed into a long format where each row represents one car at one timestamp.

This representation allows the same parameter columns to be used across cars and simplifies car-level comparison, feature engineering, and fault scoring.

In [20]:
case_01_long = wide_to_long(case_01)

print("Wide shape:", case_01.shape)
print("Long shape:", case_01_long.shape)

case_01_long.head()

Wide shape: (6999, 67)
Long shape: (55992, 12)


,Car model,Train number,Time,ACV Control Temperature (Cooling),ACV Control Temperature (Heating),Load Halved,Indoor Average Temperature,Outdoor Average Temperature,ACV Setting Mode,ACV Information Valid,ACV Running Mode,car_id
0,A,620,2023-05-18 00:00:00,25.0,20.0,Normal,25.0,27.0,Centralized Control,Valid,Automatic Cooling,01
1,A,620,2023-05-18 00:00:30,25.0,20.0,Normal,25.0,27.0,Centralized Control,Valid,Automatic Cooling,01
2,A,620,2023-05-18 00:01:00,25.0,20.0,Normal,25.0,27.0,Centralized Control,Valid,Automatic Cooling,01
3,A,620,2023-05-18 00:01:30,25.0,20.0,Normal,25.0,27.0,Centralized Control,Valid,Automatic Cooling,01
4,A,620,2023-05-18 00:02:00,25.0,20.0,Normal,24.5,27.0,Centralized Control,Valid,Automatic Cooling,01


### 2.1 Transformation Validation

Verify that the long-format dataset contains all eight car identifiers and that each car retains the same number of timestamp observations as the original wide-format dataset.

In [21]:
print("Cars:", sorted(case_01_long["car_id"].unique()))

print("\nRows per car:")
print(case_01_long["car_id"].value_counts().sort_index())

Cars: ['01', '02', '03', '04', '05', '06', '07', '08']

Rows per car:
car_id
01    6999
02    6999
03    6999
04    6999
05    6999
06    6999
07    6999
08    6999
Name: count, dtype: int64


## 3. Preprocessing Sanity Check

The long-format telemetry is passed through the reusable preprocessing pipeline. Equivalent parameter names are standardised, and observations containing no ACV telemetry are removed.

No numerical imputation or categorical encoding is performed at this stage.

In [22]:
case_01_clean = preprocess_acv(case_01_long)

print("Before preprocessing:", case_01_long.shape)
print("After preprocessing:", case_01_clean.shape)

print("\nRows per car:")
print(case_01_clean["car_id"].value_counts().sort_index())

Before preprocessing: (55992, 12)
After preprocessing: (50576, 12)

Rows per car:
car_id
01    6322
02    6322
03    6322
04    6322
05    6322
06    6322
07    6322
08    6322
Name: count, dtype: int64


### 3.1 Observation

Case 01 contains 6,999 timestamps per car before preprocessing. After removing the 677 timestamps with completely missing ACV telemetry, 6,322 observations remain for each of the eight cars.

The preprocessing pipeline therefore removes unavailable telemetry consistently across all cars while preserving the balanced temporal structure of the case.

## 4. Training Fault Labels

Each training case contains exactly one car with a documented refrigerant leakage fault. The provided training labels are loaded so that the behaviour of the faulty car can be compared against the remaining cars.

In [23]:
LABELS_PATH = (
    REPO_ROOT
    / "data"
    / "ACV"
    / "Train_Labels.csv"
)

labels = pd.read_csv(LABELS_PATH)

labels

,filename,faulty_car
0,acv_case_01.xlsx,1
1,acv_case_02.xlsx,2
2,acv_case_03.xlsx,3
3,acv_case_04.xlsx,1
4,acv_case_05.xlsx,4
5,acv_case_06.xlsx,6


### 4.1 Label Standardisation

The provided fault labels identify cars using integer values, while car identifiers extracted from the telemetry headers use two-digit strings such as `01` and `02`.

The labels are therefore converted to the same two-digit representation so that they can be matched directly with the dynamically extracted car identifiers.

In [24]:
labels["faulty_car"] = (
    labels["faulty_car"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

labels

,filename,faulty_car
0,acv_case_01.xlsx,01
1,acv_case_02.xlsx,02
2,acv_case_03.xlsx,03
3,acv_case_04.xlsx,01
4,acv_case_05.xlsx,04
5,acv_case_06.xlsx,06


## 5. Prepare Standard Training Cases for EDA

The five standard-schema training cases are loaded, transformed into long format, and preprocessed using the reusable pipeline.

Each car is then labelled as faulty or healthy according to the provided training labels. Case 04 is temporarily excluded because its substantially richer telemetry schema differs from the held-out test case.

In [25]:
standard_case_names = [
    "acv_case_01.xlsx",
    "acv_case_02.xlsx",
    "acv_case_03.xlsx",
    "acv_case_05.xlsx",
    "acv_case_06.xlsx",
]

training_cases = {}

for filename in standard_case_names:
    file_path = (
        REPO_ROOT
        / "data"
        / "ACV"
        / "Train"
        / filename
    )

    # Load → reshape → preprocess
    df = load_acv_file(file_path)
    df = wide_to_long(df)
    df = preprocess_acv(df)

    # Get the known faulty car
    faulty_car = labels.loc[
        labels["filename"] == filename,
        "faulty_car"
    ].iloc[0]

    # Attach case information and ground-truth label
    df["case"] = filename
    df["is_faulty"] = df["car_id"] == faulty_car

    training_cases[filename] = df

### 5.1 Fault-Label Validation

Verify that each training case contains exactly one unique car labelled as faulty and seven unique cars labelled as healthy.

In [26]:
for filename, df in training_cases.items():
    faulty_cars = sorted(
        df.loc[df["is_faulty"], "car_id"].unique()
    )

    healthy_cars = sorted(
        df.loc[~df["is_faulty"], "car_id"].unique()
    )

    print(f"{filename}")
    print(f"  Faulty:  {faulty_cars}")
    print(f"  Healthy: {healthy_cars}")

acv_case_01.xlsx
  Faulty:  ['01']
  Healthy: ['02', '03', '04', '05', '06', '07', '08']
acv_case_02.xlsx
  Faulty:  ['02']
  Healthy: ['01', '03', '04', '05', '06', '07', '08']
acv_case_03.xlsx
  Faulty:  ['03']
  Healthy: ['01', '02', '04', '05', '06', '07', '08']
acv_case_05.xlsx
  Faulty:  ['04']
  Healthy: ['01', '02', '03', '05', '06', '07', '08']
acv_case_06.xlsx
  Faulty:  ['06']
  Healthy: ['01', '02', '03', '04', '05', '07', '08']


## 6. Faulty vs Healthy Indoor Temperature Behaviour

Refrigerant leakage can reduce an ACV system's cooling effectiveness. A useful first investigation is therefore to compare the indoor temperature behaviour of the faulty car against the seven healthy cars within each training case.

Summary statistics are first calculated for each car before examining detailed time-series behaviour.

In [27]:
indoor_temp_summary = []

for filename, df in training_cases.items():

    for car_id in sorted(df["car_id"].unique()):
        car_data = df[df["car_id"] == car_id]

        temperatures = car_data[
            "Indoor Average Temperature"
        ].dropna()

        indoor_temp_summary.append({
            "Case": filename,
            "Car": car_id,
            "Faulty": car_data["is_faulty"].iloc[0],
            "Mean": temperatures.mean(),
            "Median": temperatures.median(),
            "Std": temperatures.std(),
            "Min": temperatures.min(),
            "Max": temperatures.max()
        })

indoor_temp_summary_df = pd.DataFrame(
    indoor_temp_summary
)

indoor_temp_summary_df.round(2)

,Case,Car,Faulty,Mean,Median,Std,Min,Max
0,acv_case_01.xlsx,01,True,24.54,24.5,2.28,0.0,32.5
1,acv_case_01.xlsx,02,False,24.19,24.0,1.67,20.0,32.5
2,acv_case_01.xlsx,03,False,24.32,24.0,1.57,20.0,32.5
3,acv_case_01.xlsx,04,False,24.25,24.0,1.54,19.5,32.0
4,acv_case_01.xlsx,05,False,24.15,24.0,1.52,19.5,31.5
5,acv_case_01.xlsx,06,False,24.17,24.0,1.55,19.5,31.5
6,acv_case_01.xlsx,07,False,24.20,24.0,1.56,19.5,31.5
7,acv_case_01.xlsx,08,False,24.15,24.0,1.46,19.5,32.0
8,acv_case_02.xlsx,01,False,23.75,24.0,1.16,21.0,28.5
9,acv_case_02.xlsx,02,True,23.91,24.0,2.08,0.0,29.0


### 6.1 Relative Indoor Temperature

Absolute temperatures can vary between cases because of different operating and environmental conditions. Therefore, each car's mean indoor temperature is compared with the average behaviour of the other cars in the same case.

A positive difference indicates that the car is warmer on average than its peers, while a negative difference indicates that it is cooler.

In [28]:
relative_temp_rows = []

for filename, case_summary in indoor_temp_summary_df.groupby("Case"):

    for _, row in case_summary.iterrows():

        peer_mean = case_summary.loc[
            case_summary["Car"] != row["Car"],
            "Mean"
        ].mean()

        relative_temp_rows.append({
            "Case": filename,
            "Car": row["Car"],
            "Faulty": row["Faulty"],
            "Mean Indoor Temp": row["Mean"],
            "Peer Mean": peer_mean,
            "Difference from Peers": (
                row["Mean"] - peer_mean
            )
        })

relative_temp_df = pd.DataFrame(relative_temp_rows)

relative_temp_df.round(2)

,Case,Car,Faulty,Mean Indoor Temp,Peer Mean,Difference from Peers
0,acv_case_01.xlsx,01,True,24.54,24.20,0.34
1,acv_case_01.xlsx,02,False,24.19,24.25,-0.06
2,acv_case_01.xlsx,03,False,24.32,24.23,0.08
3,acv_case_01.xlsx,04,False,24.25,24.24,0.00
4,acv_case_01.xlsx,05,False,24.15,24.26,-0.11
5,acv_case_01.xlsx,06,False,24.17,24.26,-0.09
6,acv_case_01.xlsx,07,False,24.20,24.25,-0.06
7,acv_case_01.xlsx,08,False,24.15,24.26,-0.11
8,acv_case_02.xlsx,01,False,23.75,23.82,-0.07
9,acv_case_02.xlsx,02,True,23.91,23.80,0.11


### 6.2 Observation

The faulty car tends to have a higher mean indoor temperature than the other cars within the same case.

In Cases 01, 03, 05, and 06, the labelled faulty car has the highest mean indoor temperature among all eight cars. In Case 02, the faulty car has an above-peer mean temperature but is slightly lower than healthy Car 03.

The magnitude of the difference also varies considerably between cases. The faulty car in Case 06 is approximately 1.33°C warmer than its peers, whereas the difference in Case 02 is only approximately 0.11°C.

Mean indoor temperature relative to the other cars therefore appears to contain useful fault-localisation information, but it is not sufficient as a standalone indicator for every case.

## 7. Baseline Fault Ranking

The initial EDA suggests that refrigerant-leakage cars tend to have higher mean indoor temperatures than their peers.

A simple baseline is therefore evaluated by ranking all eight cars within each case from highest to lowest mean indoor temperature. This provides a reference performance that subsequent feature engineering and modelling should improve upon.

In [29]:
baseline_results = []

for case, case_df in relative_temp_df.groupby("Case"):

    ranked = case_df.sort_values(
        "Mean Indoor Temp",
        ascending=False
    ).reset_index(drop=True)

    faulty_row = ranked[ranked["Faulty"]].iloc[0]

    faulty_rank = faulty_row.name + 1
    n_cars = len(ranked)

    score = (
        n_cars - (faulty_rank - 1)
    ) / n_cars

    baseline_results.append({
        "Case": case,
        "Faulty Car": faulty_row["Car"],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["Car"])
    })

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.000,01|03|04|07|02|06|05|08
1,acv_case_02.xlsx,02,2,0.875,03|02|07|06|08|04|01|05
2,acv_case_03.xlsx,03,1,1.000,03|08|04|02|01|07|06|05
3,acv_case_05.xlsx,04,1,1.000,04|02|07|01|06|03|08|05
4,acv_case_06.xlsx,06,1,1.000,06|08|04|02|03|05|01|07


### 7.1 Baseline Score

Calculate the average linear rank-decay score across the five standard-schema training cases.

In [30]:
baseline_score = baseline_results_df["Score"].mean()

print(f"Mean baseline score: {baseline_score:.3f}")

Mean baseline score: 0.975


### 7.2 Observation

Ranking cars solely by mean indoor temperature achieves a mean linear rank-decay score of **0.975** across the five standard-schema training cases.

The faulty car is ranked first in four of the five cases and second in the remaining case. This establishes a strong simple baseline and indicates that relative indoor temperature behaviour contains substantial fault-localisation information.

However, this result is calculated on the same labelled cases used during exploratory analysis and should not be interpreted as held-out model performance. Subsequent methods should be evaluated using case-level validation.

## 8. Cooling Temperature Error

Mean indoor temperature provides a strong initial localisation signal, but absolute cabin temperature alone does not indicate how effectively the ACV is responding to its cooling control target.

A temperature-error signal is therefore calculated as the difference between indoor average temperature and ACV cooling control temperature for each observation.

A larger positive value indicates that the measured indoor temperature remains further above the cooling control temperature.

In [31]:
cooling_error_summary = []

for filename, df in training_cases.items():

    df = df.copy()

    df["cooling_error"] = (
        df["Indoor Average Temperature"]
        - df["ACV Control Temperature (Cooling)"]
    )

    for car_id in sorted(df["car_id"].unique()):

        car_data = df[df["car_id"] == car_id]

        errors = car_data["cooling_error"].dropna()

        cooling_error_summary.append({
            "Case": filename,
            "Car": car_id,
            "Faulty": car_data["is_faulty"].iloc[0],
            "Mean Cooling Error": errors.mean(),
            "Median Cooling Error": errors.median(),
            "Std Cooling Error": errors.std()
        })

cooling_error_df = pd.DataFrame(
    cooling_error_summary
)

cooling_error_df.round(2)

,Case,Car,Faulty,Mean Cooling Error,Median Cooling Error,Std Cooling Error
0,acv_case_01.xlsx,01,True,0.67,0.0,1.95
1,acv_case_01.xlsx,02,False,0.20,0.0,1.33
2,acv_case_01.xlsx,03,False,0.03,0.0,1.10
3,acv_case_01.xlsx,04,False,-0.04,0.0,1.04
4,acv_case_01.xlsx,05,False,-0.13,-0.5,0.99
5,acv_case_01.xlsx,06,False,-0.12,-0.5,1.03
6,acv_case_01.xlsx,07,False,-0.09,-0.5,1.03
7,acv_case_01.xlsx,08,False,-0.14,-0.5,0.93
8,acv_case_02.xlsx,01,False,0.18,0.0,0.86
9,acv_case_02.xlsx,02,True,0.49,0.5,0.95


### 8.1 Observation

Mean cooling temperature error provides a stronger separation between faulty and healthy cars than mean indoor temperature alone.

Across all five standard-schema training cases, the labelled faulty car has the highest mean cooling error among the eight cars. This includes Case 02, where mean indoor temperature alone ranked the faulty car second.

The absolute cooling-error values vary considerably between cases and may even be negative, as observed in Case 05. Therefore, the important signal appears to be the car's relative cooling error compared with other cars in the same train rather than a universal absolute threshold.

This supports the use of peer-relative features for refrigerant-leakage localisation.

### 8.2 Cooling-Error Baseline Ranking

Cars are ranked from highest to lowest mean cooling error within each training case. The resulting ranking is evaluated using the competition's linear rank-decay metric.

In [32]:
cooling_error_results = []

for case, case_df in cooling_error_df.groupby("Case"):

    ranked = case_df.sort_values(
        "Mean Cooling Error",
        ascending=False
    ).reset_index(drop=True)

    faulty_row = ranked[ranked["Faulty"]].iloc[0]

    faulty_rank = faulty_row.name + 1
    n_cars = len(ranked)

    score = (
        n_cars - (faulty_rank - 1)
    ) / n_cars

    cooling_error_results.append({
        "Case": case,
        "Faulty Car": faulty_row["Car"],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["Car"])
    })

cooling_error_results_df = pd.DataFrame(
    cooling_error_results
)

cooling_error_results_df

,Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.0,01|02|03|04|07|06|05|08
1,acv_case_02.xlsx,02,1,1.0,02|03|07|08|06|01|04|05
2,acv_case_03.xlsx,03,1,1.0,03|02|01|07|04|08|06|05
3,acv_case_05.xlsx,04,1,1.0,04|02|07|01|06|03|08|05
4,acv_case_06.xlsx,06,1,1.0,06|08|04|02|03|05|01|07


### 8.3 Mean Cooling-Error Baseline Score

Calculate the average rank-decay score obtained by the mean cooling-error ranking.

In [33]:
cooling_error_score = cooling_error_results_df["Score"].mean()

print(
    f"Mean cooling-error baseline score: "
    f"{cooling_error_score:.3f}"
)

Mean cooling-error baseline score: 1.000


### 8.4 Cooling Error Using Valid Telemetry Only

The previous cooling-error baseline used all non-missing temperature observations. However, the dataset also contains an `ACV Information Valid` indicator, and earlier inspection showed that invalid telemetry may contain non-physical zero-valued temperatures.

The cooling-error feature is therefore recalculated using only observations where `ACV Information Valid` is `Valid`. This checks whether the observed fault-localisation pattern remains after excluding explicitly invalid telemetry.

In [34]:
valid_cooling_error_summary = []

for filename, df in training_cases.items():

    valid_df = df[
        df["ACV Information Valid"] == "Valid"
    ].copy()

    valid_df["cooling_error"] = (
        valid_df["Indoor Average Temperature"]
        - valid_df["ACV Control Temperature (Cooling)"]
    )

    for car_id in sorted(valid_df["car_id"].unique()):

        car_data = valid_df[
            valid_df["car_id"] == car_id
        ]

        errors = car_data["cooling_error"].dropna()

        valid_cooling_error_summary.append({
            "Case": filename,
            "Car": car_id,
            "Faulty": car_data["is_faulty"].iloc[0],
            "Mean Cooling Error": errors.mean()
        })

valid_cooling_error_df = pd.DataFrame(
    valid_cooling_error_summary
)

valid_cooling_error_df.round(2)

,Case,Car,Faulty,Mean Cooling Error
0,acv_case_01.xlsx,01,True,0.67
1,acv_case_01.xlsx,02,False,0.20
2,acv_case_01.xlsx,03,False,0.03
3,acv_case_01.xlsx,04,False,-0.04
4,acv_case_01.xlsx,05,False,-0.13
5,acv_case_01.xlsx,06,False,-0.12
6,acv_case_01.xlsx,07,False,-0.09
7,acv_case_01.xlsx,08,False,-0.14
8,acv_case_02.xlsx,01,False,0.18
9,acv_case_02.xlsx,02,True,0.49


### 8.5 Valid-Telemetry Cooling-Error Ranking

Cars are ranked using mean cooling error calculated only from valid ACV telemetry.

In [35]:
valid_error_results = []

for case, case_df in valid_cooling_error_df.groupby("Case"):

    ranked = case_df.sort_values(
        "Mean Cooling Error",
        ascending=False
    ).reset_index(drop=True)

    faulty_row = ranked[ranked["Faulty"]].iloc[0]

    faulty_rank = faulty_row.name + 1
    n_cars = len(ranked)

    score = (
        n_cars - (faulty_rank - 1)
    ) / n_cars

    valid_error_results.append({
        "Case": case,
        "Faulty Car": faulty_row["Car"],
        "Faulty Rank": faulty_rank,
        "Score": score,
        "Ranking": "|".join(ranked["Car"])
    })

valid_error_results_df = pd.DataFrame(valid_error_results)

valid_error_results_df

,Case,Faulty Car,Faulty Rank,Score,Ranking
0,acv_case_01.xlsx,01,1,1.0,01|02|03|04|07|06|05|08
1,acv_case_02.xlsx,02,1,1.0,02|03|07|08|06|01|04|05
2,acv_case_03.xlsx,03,1,1.0,03|02|01|07|04|08|06|05
3,acv_case_05.xlsx,04,1,1.0,04|02|07|01|06|03|08|05
4,acv_case_06.xlsx,06,1,1.0,06|08|04|02|03|05|01|07


In [36]:
print(
    "Mean valid-telemetry score:",
    round(valid_error_results_df["Score"].mean(), 3)
)

Mean valid-telemetry score: 1.0


### 8.6 Observation

After restricting the analysis to telemetry explicitly marked as `Valid`, mean cooling error continues to rank the labelled faulty car first in all five standard-schema training cases, producing a mean rank-decay score of **1.000**.

This indicates that the observed cooling-error pattern is not dependent on explicitly invalid telemetry. Compared with mean indoor temperature alone, which ranked the faulty car second in Case 02, cooling error provides stronger separation across the available standard training cases.

The result supports mean cooling error as a primary candidate feature for refrigerant-leakage localisation. However, because the same five labelled cases were used to identify and evaluate this relationship, the score should be treated as an exploratory result rather than an estimate of unseen-case performance.

## 9. Persistence of Peer-Relative Cooling Error

Mean cooling error successfully localises the faulty car in all five standard training cases. However, an overall mean can potentially be influenced by a relatively small number of extreme observations.

To examine whether the anomaly is persistent, each car's cooling error is compared with the other cars at the same timestamp. The proportion of timestamps at which each car has the highest cooling error is then calculated.

A persistently faulty ACV system would be expected to exhibit abnormal cooling behaviour repeatedly rather than only during isolated observations.

In [37]:
persistence_rows = []

for filename, df in training_cases.items():

    valid_df = df[
        df["ACV Information Valid"] == "Valid"
    ].copy()

    valid_df["cooling_error"] = (
        valid_df["Indoor Average Temperature"]
        - valid_df["ACV Control Temperature (Cooling)"]
    )

    error_matrix = valid_df.pivot(
        index="Time",
        columns="car_id",
        values="cooling_error"
    )

    highest_error_car = error_matrix.idxmax(
        axis=1,
        skipna=True
    )

    proportions = (
        highest_error_car
        .value_counts(normalize=True)
    )

    faulty_car = labels.loc[
        labels["filename"] == filename,
        "faulty_car"
    ].iloc[0]

    for car_id in sorted(df["car_id"].unique()):
        persistence_rows.append({
            "Case": filename,
            "Car": car_id,
            "Faulty": car_id == faulty_car,
            "Highest Error (%)": (
                proportions.get(car_id, 0) * 100
            )
        })

persistence_df = pd.DataFrame(persistence_rows)

persistence_df.round(2)

,Case,Car,Faulty,Highest Error (%)
0,acv_case_01.xlsx,01,True,54.60
1,acv_case_01.xlsx,02,False,18.90
2,acv_case_01.xlsx,03,False,8.45
3,acv_case_01.xlsx,04,False,4.95
4,acv_case_01.xlsx,05,False,2.88
5,acv_case_01.xlsx,06,False,3.37
6,acv_case_01.xlsx,07,False,3.05
7,acv_case_01.xlsx,08,False,3.80
8,acv_case_02.xlsx,01,False,26.87
9,acv_case_02.xlsx,02,True,40.73


In [38]:
persistence_ranking = (
    persistence_df
    .sort_values(
        ["Case", "Highest Error (%)"],
        ascending=[True, False]
    )
)

persistence_ranking.groupby("Case").head(3).round(2)

,Case,Car,Faulty,Highest Error (%)
0,acv_case_01.xlsx,01,True,54.60
1,acv_case_01.xlsx,02,False,18.90
2,acv_case_01.xlsx,03,False,8.45
9,acv_case_02.xlsx,02,True,40.73
8,acv_case_02.xlsx,01,False,26.87
10,acv_case_02.xlsx,03,False,11.72
18,acv_case_03.xlsx,03,True,42.67
16,acv_case_03.xlsx,01,False,28.40
17,acv_case_03.xlsx,02,False,13.66
24,acv_case_05.xlsx,01,False,40.62


### 9.1 Observation

The faulty car exhibits the highest cooling error most frequently in Cases 01, 02, 03, and 06. In these cases, the faulty car has the largest instantaneous cooling error at approximately 41–87% of evaluated timestamps.

Case 05 behaves differently. The faulty Car 04 has the highest instantaneous cooling error at only 15.20% of timestamps and ranks behind healthy Cars 01 and 02 by this persistence measure.

Therefore, persistence of the maximum instantaneous cooling error contains useful fault-localisation information but is not sufficient as a standalone indicator. In particular, Case 05 demonstrates that a car can have the highest overall mean cooling error without being the most abnormal car at the largest number of individual timestamps.

## 10. EDA Conclusions

Exploratory analysis of the five standard-schema training cases identified cooling-performance behaviour as a strong candidate signal for refrigerant-leakage localisation.

The main findings are:

- Mean indoor temperature alone provides a strong baseline, ranking the faulty car first in four of five cases and second in one case.
- Mean cooling error, defined as indoor average temperature minus ACV cooling control temperature, ranks the faulty car first in all five standard training cases.
- The same result is retained when the calculation is restricted to telemetry explicitly marked as `Valid`.
- Comparing cars within the same train appears more informative than applying universal temperature thresholds because operating conditions differ between cases.
- Persistence of the highest instantaneous cooling error provides additional information, although Case 05 demonstrates that it is not reliable as a standalone ranking rule.

These findings motivate a feature representation based primarily on each car's cooling performance relative to both its control temperature and the behaviour of the other cars in the same case.

The next stage converts these observations into reusable car-level features and evaluates their ability to generalise across fault cases using case-level validation.